In [7]:
import pandas as pd
import matplotlib.pyplot as plt

In [8]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def make_and_save_plots(analysis_name):
    min_font_size = 14
    
    root_path = "../.."
    folder_path=root_path+"/results/"+analysis_name+"/"

    results_df = pd.read_csv(folder_path+analysis_name+"_analysis.csv")

    unique_name = results_df['name'].unique()

    # Find unique difftime values to create a subplot for each
    unique_difftimes = results_df['difftime'].unique()
    num_plots = len(unique_difftimes)
    unique_letters = ["a", "b", "c", "d"]

    # Create a subplot figure
    fig = make_subplots(
        rows=1, cols=num_plots,
        specs=[[{'type': 'scatter3d'}] * num_plots],
        subplot_titles=[f"{l}) t<sub>hy</sub> = {d} s" for d, l in zip(unique_difftimes, unique_letters)],
        horizontal_spacing=0,  # Set horizontal spacing to 0
        vertical_spacing=0     # Set vertical spacing to 0
    )

    min_vfinal = results_df['Vfinal'].min()
    max_vfinal = results_df['Vfinal'].max()
    max_vfinal = 1e0

    # Iterate and add a trace for each difftime value
    for i, d in enumerate(unique_difftimes, start=1):
        subset_df = results_df[results_df["difftime"] == d].copy()
        
        # 1. Split the data based on your condition
        match_mask = subset_df['goodmatch'] == "yes"
        df_normal = subset_df[~match_mask]
        df_match = subset_df[match_mask]
        
        # 2. Add Trace for NORMAL points
        if not df_normal.empty:
            trace_normal = go.Scatter3d(
                x=df_normal['alpha'],
                y=df_normal['epsilon'],
                z=df_normal['beta'],
                mode='markers',
                marker=dict(
                    size=4,
                    color=np.log10(df_normal['Vfinal']),
                    colorscale='Viridis',
                    colorbar=dict(
                        title='log<sub>10</sub>(V<sub>final</sub>/V<sub>0</sub>)',
                    ),
                    cmin=np.log10(min_vfinal),
                    cmax=np.log10(max_vfinal),
                    line=dict(width=0)
                )
            )
            fig.add_trace(trace_normal, row=1, col=i)
        
        # 3. Add Trace for GOODMATCH points (with the thick line)
        if not df_match.empty:
            trace_match = go.Scatter3d(
                x=df_match['alpha'],
                y=df_match['epsilon'],
                z=df_match['beta'],
                mode='markers',
                marker=dict(
                    size=4,
                    color=np.log10(df_match['Vfinal']), 
                    colorscale='Viridis',
                    showscale=False,
                    cmin=np.log10(min_vfinal),
                    cmax=np.log10(max_vfinal),
                    line=dict(
                        width=1,
                        color='black'
                    )
                )
            )
            fig.add_trace(trace_match, row=1, col=i)
        
        # Set axes to log10
        fig.update_scenes(
            xaxis_title="Alpha",
            yaxis=dict(type='log', title='Epsilon', tickformat=".0e", nticks=4),
            zaxis=dict(type='log', title='Beta', tickformat=".0e", nticks=4),
            row=1, col=i,
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=1.5)
            )
        )

    #subplots
    fig.update_annotations(
    font=dict(
        family="Arial, sans-serif",
        size=min_font_size+12,
        color="black"
    )
)
    fig.update_layout(
        title = dict(
            text=unique_name[0],
            font=dict(family="Arial, sans-serif", size=min_font_size+12)
        ),
        height=600,
        showlegend=False,
        width=2000,
        font=dict(
            family="Arial, sans-serif",
            size=min_font_size,
            color="black"
        )
    )

    fig.show()

    # Save the figure
    fig.write_image(folder_path+analysis_name+"_analysis.png",scale=2)
    fig.write_image(folder_path+analysis_name+"_analysis.svg",scale=2)
    fig.write_image(folder_path+analysis_name+"_analysis.pdf",scale=2)

In [9]:
for analysis_name in ["gsa-shiva-bg","gsa-shiva-bg-llf","gsa-shiva-bg-mlf", "gsa-shiva-bg-lf","gsa-shiva-mcf","gsa-brava-mcf"]:
    make_and_save_plots(analysis_name)